# python cheatsheet interactive: practice

An agent loop must make progress and stop → Streaming sends partial progress

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [guides/community_contributions/python_cheatsheet_interactive.ipynb](../../../../../../../guides/community_contributions/python_cheatsheet_interactive.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. An agent loop must make progress and stop

A loop repeats model/tool steps until a final answer or a configured limit. This fixed script of decisions shows the order without executing generated code or calling a model.

**Predict:** Which decision ends the loop?

In [ ]:
decisions = iter([
    {"kind": "tool", "name": "add", "a": 12, "b": 30},
    {"kind": "answer", "text": "The sum is 42."},
])
events = []
for turn in range(3):  # Bound the number of steps.
    decision = next(decisions)
    if decision["kind"] == "tool":
        if decision["name"] != "add":
            raise ValueError("Unknown tool")
        events.append({"tool_result": decision["a"] + decision["b"]})
    else:
        events.append({"final": decision["text"]})
        break
else:
    raise RuntimeError("No final answer within the limit")
print(events)
assert events[0]["tool_result"] == 42

**Check:** The answer decision ends it. A loop that only keeps requesting tools needs a bound and an explicit failure path.

**Change one thing:** Add another tool step before the answer. Keep a finite turn limit and explain how the stopping rule changes.

## 2. Streaming sends partial progress

A generator yields multiple updates. A streamed API can include multiple event types, so a UI should select the text events rather than assume every event is text.

**Predict:** How many text chunks are selected?

In [ ]:
events = [
    {"type": "status", "value": "started"},
    {"type": "text", "value": "Hello "},
    {"type": "text", "value": "learner"},
]
def text_chunks(items):
    for event in items:
        if event["type"] == "text":
            yield event["value"]
chunks = list(text_chunks(events))
print(chunks, "→", "".join(chunks))
assert "".join(chunks) == "Hello learner"

**Check:** Two. yield preserves progress; return ends an ordinary function.

**Change one thing:** Insert a non-text event between the chunks. Confirm it does not become displayed answer text.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.